<a href="https://colab.research.google.com/github/nhemanth1103/AI-Powered-Smart-Email-Classifier-for-Enterprises/blob/main/Emails.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import Libraries


In [1]:
import pandas as pd
import numpy as np
import joblib
import re


#sklearn imports
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

Load the data

In [2]:
df = pd.read_csv("merged_data.csv")

print(f"Dataset Loaded: {df.shape[0]} emails")

Dataset Loaded: 204907 emails


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 204907 entries, 0 to 204906
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    204907 non-null  object
 1   label   204907 non-null  object
dtypes: object(2)
memory usage: 3.1+ MB


In [4]:
#columns_name
df.columns

Index(['text', 'label'], dtype='object')

In [5]:
df.head()

,text,label
0,what a wonderful spice blend it is generous an...,feedback
1,i have a question about account hacked relatin...,support
2,how do i proceed with sensor offline given tha...,support
3,regarding our chat earlier about soundtrack sc...,other
4,i have been using the upcycled denim for a whi...,feedback


In [6]:
#No. of labels
df['label'].value_counts()

,count
label,
spam,43132
feedback,41408
other,40647
complaint,40056
support,39664


Build the pipeline

In [7]:
#vectorizer
vectorizer = TfidfVectorizer(stop_words = 'english', max_features = 8000, ngram_range=(1,3))

#Linear SVM
svm = LinearSVC(class_weight='balanced', random_state=42, max_iter=10000)

# Step C: Probability Calibrator
calibrated_svm = CalibratedClassifierCV(svm, method='sigmoid', cv=3)

pipeline = Pipeline([
    ('tfidf', vectorizer),
    ('clf', calibrated_svm)
])

Train the model

In [8]:
X = df['text']
y = df['label']

#split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=8000, ngram_range=(1, 3),
                                 stop_words='english')),
                ('clf',
                 CalibratedClassifierCV(cv=3,
                                        estimator=LinearSVC(class_weight='balanced',
                                                            max_iter=10000,
                                                            random_state=42)))])

Evaluate

In [9]:
y_pred = pipeline.predict(X_test)
print(f"Model Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Model Accuracy: 99.22%

Classification Report:

              precision    recall  f1-score   support

   complaint       0.99      0.99      0.99      8011
    feedback       0.99      0.99      0.99      8282
       other       0.99      0.99      0.99      8129
        spam       1.00      1.00      1.00      8627
     support       0.99      1.00      1.00      7933

    accuracy                           0.99     40982
   macro avg       0.99      0.99      0.99     40982
weighted avg       0.99      0.99      0.99     40982



Words that carries weight

In [10]:
def get_svm_keywords(text, pipeline, top_n=3):
  model = pipeline.named_steps['clf']

  base_svm = model.calibrated_classifiers_[0].estimator

  vec = pipeline.named_steps['tfidf']

  pred_class = pipeline.predict([text])[0]

  class_idx = list(model.classes_).index(pred_class)

  if base_svm.coef_.shape[0] == 1:
        # Binary case
        coefs = base_svm.coef_[0] if class_idx == 1 else -base_svm.coef_[0]
  else:
        # Multi-class case
        coefs = base_svm.coef_[class_idx]

  feature_names = vec.get_feature_names_out()

  input_words = re.findall(r'\w+', text.lower())
  word_scores = []

  # Create a quick lookup for efficiency
  vocab = vec.vocabulary_

  for word in input_words:
      if word in vocab:
          idx = vocab[word]
          score = coefs[idx]
          if score > 0: # Only positive contributors
              word_scores.append((word, score))

  # Sort by impact
  word_scores.sort(key=lambda x: x[1], reverse=True)
  return [w[0] for w in word_scores[:top_n]]

# Test Explainability
test_email = "Urgent: System crash! Login failed for all users."
print(f"\nTest Email: {test_email}")
print(f"Prediction: {pipeline.predict([test_email])[0]}")
print(f"Key Drivers: {get_svm_keywords(test_email, pipeline)}")


Test Email: Urgent: System crash! Login failed for all users.
Prediction: support
Key Drivers: ['crash', 'failed', 'login']


Save the Model

In [11]:
joblib.dump(pipeline, 'svm_model.pkl')

['svm_model.pkl']